<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-08-agents-and-adk/lesson-8.1-root-agent/practice/GCP_Capstone_8.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 8.1 — Root Agent with ADK

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup (run first)

Installs the ADK + unified GenAI SDK, authenticates with **Application Default Credentials** (never API keys), and points ADK at Vertex AI. Run this once before any exercise.

In [ ]:
!pip install -q google-adk google-genai mcp
import os
from google.colab import auth
auth.authenticate_user()

# Application Default Credentials -> Vertex AI (no API keys)
os.environ['GOOGLE_CLOUD_PROJECT']  = 'documind-ai-YOUR-ID'
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'  # global endpoint for Gemini 3.x generation
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
print('ADK ready (Vertex + ADC)')

## Exercise 1: First LlmAgent

**Difficulty:** Easy

Create `root_agent` with no tools and test that it responds. In the full workflow you would test with `adk web`; here in Colab we test in-process with a tiny `Runner` helper.

**Steps**
1. Import `LlmAgent` and `types`.
2. Build a `root_agent` (model `gemini-3.6-flash`) with an instruction but **no tools**.
3. Define a reusable `run_turns()` helper (used by later exercises to see tool calls + state).
4. Send a greeting and confirm the agent replies.

**Expected behaviour:** Agent responds to greetings.

In [ ]:
from google.adk.agents import LlmAgent
from google.genai import types

# NO generate_content_config here: it held only temperature, and gemini-3.x
# ignores temperature, top_p and top_k. A config that changes nothing is worse
# than no config - it reads like a lever somebody can pull.

root_agent = LlmAgent(
    name='documind',
    model='gemini-3.6-flash',
    instruction='You are DocuMind AI, a helpful document assistant. Greet users warmly '
                'and briefly explain what you can help with.',
)
print(f'Agent: {root_agent.name} (no tools)')

In [ ]:
# Reusable in-process test harness (used by later exercises too).
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

async def run_turns(agent, messages, app='dm', user='s1'):
    ss = InMemorySessionService()
    runner = Runner(agent=agent, app_name=app, session_service=ss)
    session = await ss.create_session(app_name=app, user_id=user)
    for msg in messages:
        print(f'\nUser: {msg}')
        m = Content(role='user', parts=[Part(text=msg)])
        async for ev in runner.run_async(user_id=user, session_id=session.id, new_message=m):
            for p in (ev.content.parts if ev.content else []):
                fc = getattr(p, 'function_call', None)
                if fc:
                    print(f'  [tool call] {fc.name}({dict(fc.args)})')
            if ev.is_final_response() and ev.content and ev.content.parts:
                for p in ev.content.parts:
                    if p.text:
                        print(f'Agent: {p.text[:200]}')
    return await ss.get_session(app_name=app, user_id=user, session_id=session.id)

await run_turns(root_agent, ['Hi there! What can you do?'])

## Exercise 2: Add One Tool

**Difficulty:** Easy

Add `retrieve` and verify the tool call is visible in the event stream.

**Steps**
1. Import `ToolContext` and define `retrieve(query, tool_context)` with a clear docstring.
2. Build an agent that registers just this one tool.
3. Ask a question that forces a search and watch for the `[tool call]` line.

**Expected behaviour:** Tool call visible in events.

In [ ]:
from google.adk.tools import ToolContext

# DocuMind's corpus - the same six chunks the main lesson uses, so an
# answer here and an answer there cite the same page of the same PDF.
GS = "gs://documind-acme"
_DOCS = [
    # doc_id, doc_type, page, score, quote
    ("hr_policy_2026", "policy", 12, 0.94,
     "A senior engineer serves a notice period of 60 days."),
    ("hr_policy_2026", "policy", 31, 0.81,
     "Earned leave is encashed on exit, capped at 45 days."),
    ("msa_acme_2026", "contract", 8, 0.88,
     "Either party may terminate on 90 days written notice."),
    ("inv_2026_0412", "invoice", 1, 0.76,
     "Total payable Rs 1,84,500, inclusive of 18% GST."),
    ("gstr1_q1_fy27", "form", 4, 0.68,
     "Outward taxable supplies for the quarter, GSTR-1."),
    ("rag_survey_2026", "research_paper", 6, 0.72,
     "Hybrid retrieval mixes dense and sparse signals."),
]
CORPUS = [{"chunk_id": f"{d}#{p}", "doc_type": t, "page": p,
           "source_uri": f"{GS}/{d}.pdf", "quote": q,
           "score": s} for d, t, p, s, q in _DOCS]
CITATION_FIELDS = ("chunk_id", "source_uri", "page", "quote", "score")


def retrieve(query: str, doc_type: str = "all", top_k: int = 5,
             tool_context: ToolContext = None) -> dict:
    """Retrieve grounded passages from DocuMind's corpus.

    Args:
        query: The question, in natural language
        doc_type: policy, contract, invoice, form,
            research_paper, or all
        top_k: How many passages to return
    """
    if tool_context is not None:
        # Session state, not a global: this is the ADK seam, and it is
        # why the tool takes a context rather than reaching for a
        # module-level variable.
        history = tool_context.state.get("search_history", [])
        history.append(query)
        tool_context.state["search_history"] = history
    # A mock, but not a stub: it really filters and ranks, so a
    # question the corpus cannot answer returns NOTHING and
    # answerable=False. A mock that always succeeds teaches that
    # retrieval always succeeds - the one thing it never does.
    words = {w for w in query.lower().split() if len(w) > 3}
    hits = [c for c in CORPUS
            if doc_type in ("all", c["doc_type"])
            and any(w in c["quote"].lower() for w in words)]
    hits.sort(key=lambda c: -c["score"])
    hits = hits[:top_k]
    top = hits[0]["score"] if hits else 0.0
    return {
        "citations": [{k: c[k] for k in CITATION_FIELDS} for c in hits],
        "answerable": bool(hits),
        "confidence": ("high" if top >= 0.85 else
                       "medium" if hits else "low"),
    }
agent_search = LlmAgent(
    name='documind', model='gemini-3.6-flash',
    instruction='You are DocuMind AI. Use retrieve to find documents.',
    tools=[retrieve],
)
await run_turns(agent_search, ['Find the Q1 financial report'])

## Exercise 3: Project Structure

**Difficulty:** Easy

Create the agent package (`__init__.py` + `agent.py` + `.env`) so `adk web` can discover it.

**Steps**
1. Make a `documind_agent/` directory.
2. Write `__init__.py` that imports `agent`.
3. Write `agent.py` exposing a `root_agent`.
4. Write `.env` with the **Vertex + ADC** settings (no API key).
5. Launch `adk web` from the parent directory.

**Expected behaviour:** Agent discovered by `adk web`.

*Note: the lesson notebook's `.env` used `GOOGLE_API_KEY`; we use Vertex/ADC settings to match the course's ADC-only convention.*

In [ ]:
import os
os.makedirs('documind_agent', exist_ok=True)

with open('documind_agent/__init__.py', 'w') as f:
    f.write('from . import agent\n')

agent_py = '''from google.adk.agents import LlmAgent
from google.adk.tools import ToolContext
from google.genai import types

# DocuMind's corpus - the same six chunks the main lesson uses, so an
# answer here and an answer there cite the same page of the same PDF.
GS = "gs://documind-acme"
_DOCS = [
    # doc_id, doc_type, page, score, quote
    ("hr_policy_2026", "policy", 12, 0.94,
     "A senior engineer serves a notice period of 60 days."),
    ("hr_policy_2026", "policy", 31, 0.81,
     "Earned leave is encashed on exit, capped at 45 days."),
    ("msa_acme_2026", "contract", 8, 0.88,
     "Either party may terminate on 90 days written notice."),
    ("inv_2026_0412", "invoice", 1, 0.76,
     "Total payable Rs 1,84,500, inclusive of 18% GST."),
    ("gstr1_q1_fy27", "form", 4, 0.68,
     "Outward taxable supplies for the quarter, GSTR-1."),
    ("rag_survey_2026", "research_paper", 6, 0.72,
     "Hybrid retrieval mixes dense and sparse signals."),
]
CORPUS = [{"chunk_id": f"{d}#{p}", "doc_type": t, "page": p,
           "source_uri": f"{GS}/{d}.pdf", "quote": q,
           "score": s} for d, t, p, s, q in _DOCS]
CITATION_FIELDS = ("chunk_id", "source_uri", "page", "quote", "score")


def retrieve(query: str, doc_type: str = "all", top_k: int = 5,
             tool_context: ToolContext = None) -> dict:
    """Retrieve grounded passages from DocuMind's corpus.

    Args:
        query: The question, in natural language
        doc_type: policy, contract, invoice, form,
            research_paper, or all
        top_k: How many passages to return
    """
    if tool_context is not None:
        # Session state, not a global: this is the ADK seam, and it is
        # why the tool takes a context rather than reaching for a
        # module-level variable.
        history = tool_context.state.get("search_history", [])
        history.append(query)
        tool_context.state["search_history"] = history
    # A mock, but not a stub: it really filters and ranks, so a
    # question the corpus cannot answer returns NOTHING and
    # answerable=False. A mock that always succeeds teaches that
    # retrieval always succeeds - the one thing it never does.
    words = {w for w in query.lower().split() if len(w) > 3}
    hits = [c for c in CORPUS
            if doc_type in ("all", c["doc_type"])
            and any(w in c["quote"].lower() for w in words)]
    hits.sort(key=lambda c: -c["score"])
    hits = hits[:top_k]
    top = hits[0]["score"] if hits else 0.0
    return {
        "citations": [{k: c[k] for k in CITATION_FIELDS} for c in hits],
        "answerable": bool(hits),
        "confidence": ("high" if top >= 0.85 else
                       "medium" if hits else "low"),
    }
def summarize_document(document_id: str, summary_type: str, tool_context: ToolContext) -> dict:
    """Summarize a document.
    Args:
        document_id: Doc ID.
        summary_type: brief/detailed/executive.
    """
    tool_context.state["last_summarized"] = document_id
    return {"summary": f"Summary of {document_id}"}

def calculate_cost(page_count: int, tier: str) -> dict:
    """Calculate processing cost.
    Args:
        page_count: Pages.
        tier: standard/premium/enterprise.
    """
    rates = {"standard": 0.01, "premium": 0.03, "enterprise": 0.05}
    return {"cost_usd": round(page_count * rates.get(tier, 0.01), 2)}

# NO generate_content_config here: it held only temperature, and gemini-3.x
# ignores temperature, top_p and top_k. A config that changes nothing is worse
# than no config - it reads like a lever somebody can pull.

root_agent = LlmAgent(
    name="documind", model="gemini-3.6-flash",
    instruction="You are DocuMind AI. Search before summarizing.",
    tools=[retrieve, summarize_document, calculate_cost],
)
'''
with open('documind_agent/agent.py', 'w') as f:
    f.write(agent_py)

with open('documind_agent/.env', 'w') as f:
    f.write('GOOGLE_GENAI_USE_VERTEXAI=TRUE\n')
    f.write('GOOGLE_CLOUD_PROJECT=documind-ai-YOUR-ID\n')
    f.write('GOOGLE_CLOUD_LOCATION=global\n')

print('Agent package created: documind_agent/ (__init__.py, agent.py, .env)')

**Run this in a separate terminal / Cloud Shell** — `adk web` starts a long-running server that would block the notebook kernel:

```bash
# Launch the ADK dev playground; it auto-discovers documind_agent/. Open the printed URL.
adk web
```

## Exercise 4: Three Tools

**Difficulty:** Medium

Register `retrieve`, `summarize_document`, and `calculate_cost`, then test that the agent picks the correct tool per query.

**Steps**
1. Define `summarize_document` and `calculate_cost` (search is already defined).
2. Build an agent registering all three tools.
3. Send one query per tool and confirm the matching `[tool call]`.

**Expected behaviour:** Correct tool selected per query.

In [ ]:
def summarize_document(document_id: str, summary_type: str, tool_context: ToolContext) -> dict:
    """Summarize a specific document.
    Args:
        document_id: Document ID.
        summary_type: brief, detailed, or executive.
    """
    tool_context.state['last_summarized'] = document_id
    return {'summary': f'Summary of {document_id}...'}

def calculate_cost(page_count: int, tier: str) -> dict:
    """Calculate document processing cost.
    Args:
        page_count: Number of pages.
        tier: standard, premium, enterprise.
    """
    rates = {'standard': 0.01, 'premium': 0.03, 'enterprise': 0.05}
    return {'cost_usd': round(page_count * rates.get(tier, 0.01), 2)}

# NO generate_content_config here: it held only temperature, and gemini-3.x
# ignores temperature, top_p and top_k. A config that changes nothing is worse
# than no config - it reads like a lever somebody can pull.

agent_three = LlmAgent(
    name='documind', model='gemini-3.6-flash',
    instruction='You are DocuMind AI. Use retrieve to find docs. '
                'Use summarize_document for summaries. Use calculate_cost for pricing.',
    tools=[retrieve, summarize_document, calculate_cost],
)

await run_turns(agent_three, [
    'Find financial documents',
    'Give me a brief summary of document D-01',
    'What does 100 pages cost at the premium tier?',
])

## Exercise 5: ToolContext State

**Difficulty:** Medium

Track `search_history` in `ToolContext.state` and verify it accumulates across turns within one session.

**Steps**
1. Reuse the three-tool agent (its `retrieve` already appends to `search_history`).
2. Run two searches in the same session.
3. Read the session state back and print `search_history`.

**Expected behaviour:** State accumulates across turns.

In [ ]:
session = await run_turns(agent_three, [
    'Search for revenue documents',
    'Now search for expense documents',
])
print('\nsearch_history:', dict(session.state).get('search_history'))

## Exercise 6: Instruction Design

**Difficulty:** Medium

Write an instruction that enforces "search before summarize" and verify the agent follows it — even when asked to summarize directly.

**Steps**
1. Build an agent whose instruction ends with *"Always search before summarizing."*
2. Ask it to summarize a topic without first searching.
3. Confirm the `[tool call]` order shows `retrieve` before `summarize_document`.

**Expected behaviour:** Agent searches first.

In [ ]:
# NO generate_content_config here: it held only temperature, and gemini-3.x
# ignores temperature, top_p and top_k. A config that changes nothing is worse
# than no config - it reads like a lever somebody can pull.
agent_ordered = LlmAgent(
    name='documind', model='gemini-3.6-flash',
    instruction='You are DocuMind AI. Use retrieve to find docs. '
                'Use summarize_document for summaries. Use calculate_cost for pricing. '
                'Always search before summarizing.',
    tools=[retrieve, summarize_document, calculate_cost],
)

await run_turns(agent_ordered, ['Summarize the latest quarterly report for me'])

## Exercise 7: MCP + Python

**Difficulty:** Challenge

Combine plain Python function tools with an `McpToolset` in a single agent — the pattern from the main lesson's Step 5.

**Steps**
1. Keep your existing Python tools (`retrieve`, `summarize_document`, `calculate_cost`).
2. Add an `McpToolset` connected to an MCP server over stdio.
3. Register both tool types in one `LlmAgent`.

**Expected behaviour:** Both tool types work in one agent.

*The lesson notebook covers only Python tools; this MCP wiring is illustrative and needs a real MCP server (e.g. the filesystem server) to actually run. Import paths follow current `google-adk`.*

In [ ]:
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters

# MCP filesystem server over stdio (needs Node/npx available at runtime).
mcp_tools = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command='npx',
            args=['-y', '@modelcontextprotocol/server-filesystem', '/content'],
        ),
    ),
)

# NO generate_content_config here: it held only temperature, and gemini-3.x
# ignores temperature, top_p and top_k. A config that changes nothing is worse
# than no config - it reads like a lever somebody can pull.

hybrid_agent = LlmAgent(
    name='documind', model='gemini-3.6-flash',
    instruction='You are DocuMind AI. Use retrieve / summarize_document / '
                'calculate_cost for document work, and the filesystem MCP tools to '
                'read or list files when asked.',
    tools=[retrieve, summarize_document, calculate_cost, mcp_tools],
)
print(f'Hybrid agent ready: Python tools + MCP toolset ({hybrid_agent.name})')

## Exercise 8: Runner Script

**Difficulty:** Challenge

Write a standalone `Runner` script: create a session, send 3 turns, and print the events (including the final state).

**Steps**
1. Build a `Runner` over an `InMemorySessionService`.
2. Create one session and loop over three user messages.
3. Print each agent final response, then dump the accumulated session state.

**Expected behaviour:** A 3-turn conversation with state at the end.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

async def test_agent():
    ss = InMemorySessionService()
    runner = Runner(agent=agent_three, app_name='dm', session_service=ss)
    session = await ss.create_session(app_name='dm', user_id='s1')
    for msg in ['Find financial documents',
                'Summarize the first result',
                'Cost for 100 pages premium?']:
        print(f'\nUser: {msg}')
        m = Content(role='user', parts=[Part(text=msg)])
        async for ev in runner.run_async(user_id='s1', session_id=session.id, new_message=m):
            if ev.is_final_response() and ev.content and ev.content.parts:
                for p in ev.content.parts:
                    if p.text:
                        print(f'Agent: {p.text[:150]}')
    s = await ss.get_session(app_name='dm', user_id='s1', session_id=session.id)
    print(f'\nState: {dict(s.state)}')

await test_agent()